In [ ]:
# In[1]:

import pandas as pd

# Read CSVs and parse timestamps as UTC
metric_df = pd.read_csv("metric.csv")
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

trace_df = pd.read_csv("trace.csv")
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

log_df = pd.read_csv("log.csv")
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

error_df = pd.read_csv("error_logs.csv")
error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# Ensure 'value' columns are numeric where present (coerce non-numeric to NaN)
if 'value' in metric_df.columns:
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
if 'value' in trace_df.columns:
    trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
if 'value' in log_df.columns:
    log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# Aggregation helper
def summarize(df, group_cols, value_col):
    # work on a copy column that is numeric
    numeric = pd.to_numeric(df[value_col], errors='coerce')
    g = df.assign(__val=numeric).groupby(group_cols)['__val']
    agg = g.agg(count='count', min_val='min', max_val='max', mean_val='mean')
    p95 = g.quantile(0.95).rename('p95(value)')
    agg = agg.join(p95).reset_index()
    # rename columns to requested names
    agg = agg.rename(columns={'min_val': 'min(value)', 'max_val': 'max(value)', 'mean_val': 'mean(value)'})
    agg = agg.sort_values('count', ascending=False)
    return agg

# Compute summaries and limit to top 50 rows each
metric_summary = summarize(metric_df, ['cmdb_id', 'kpi_name'], 'value').head(50)
trace_summary  = summarize(trace_df,   ['cmdb_id', 'trace_name'], 'value').head(50)
log_summary    = summarize(log_df,     ['cmdb_id', 'log_name'], 'value').head(50)

# Unique cmdb_id values across all four files
cmdb_ids = sorted(
    set(metric_df['cmdb_id'].dropna().unique()) |
    set(trace_df['cmdb_id'].dropna().unique()) |
    set(log_df['cmdb_id'].dropna().unique()) |
    set(error_df['cmdb_id'].dropna().unique())
)
cmdb_ids_series = pd.Series(cmdb_ids, name='cmdb_id')

# Return compact results: top summaries and cmdb_id list
metric_summary, trace_summary, log_summary, cmdb_ids_series

```
Out[1]:
```
```python
# Summarize findings from previous aggregations (variables exist in the kernel state)
# Uses: metric_summary, trace_summary, log_summary, cmdb_ids_series

def get_stats(df, cmdb, name_col, name):
    sub = df[(df['cmdb_id']==cmdb) & (df[name_col]==name)]
    if sub.empty:
        return None
    row = sub.iloc[0]
    return row['mean(value)'], row['max(value)'], int(row['count'])

# Check trace
trace_empty = trace_summary.empty if 'trace_summary' in globals() else True

# Important observations for 'carts'
carts_cpu = get_stats(metric_summary, 'carts', 'kpi_name', 'cpu')
carts_mem = get_stats(metric_summary, 'carts', 'kpi_name', 'mem')
carts_socket = get_stats(metric_summary, 'carts', 'kpi_name', 'socket')
carts_rowcount = get_stats(log_summary, 'carts', 'log_name', 'log.row_count')

# Catalogue log traffic
catalogue_rowcount = get_stats(log_summary, 'catalogue', 'log_name', 'log.row_count')
catalogue_error = get_stats(log_summary, 'catalogue', 'log_name', 'log.error_count')

# Rabbitmq quick note
rabbit_cpu = get_stats(metric_summary, 'rabbitmq', 'kpi_name', 'cpu')
rabbit_socket = get_stats(metric_summary, 'rabbitmq', 'kpi_name', 'socket')

# Unique cmdb count
num_cmdb = len(cmdb_ids_series) if 'cmdb_ids_series' in globals() else 0

# Build summary string
parts = []
parts.append(f"Aggregations computed for metric, trace, and log telemetry across {num_cmdb} unique cmdb_id(s).")
parts.append("Trace data: no trace-series rows were found." if trace_empty else "Trace data: present.")

# carts observations
if carts_cpu or carts_mem or carts_socket or carts_rowcount:
    parts.append("Notable signals for 'carts':")
    if carts_cpu:
        parts.append(f" - CPU (count={carts_cpu[2]}): mean≈{carts_cpu[0]:.2f}, max≈{carts_cpu[1]:.2f}")
    if carts_mem:
        parts.append(f" - Memory (count={carts_mem[2]}): mean≈{carts_mem[0]:.0f}, max≈{carts_mem[1]:.0f}")
    if carts_socket:
        parts.append(f" - Socket (count={carts_socket[2]}): mean≈{carts_socket[0]:.2f}, max≈{carts_socket[1]:.2f}")
    if carts_rowcount:
        parts.append(f" - log.row_count (count={carts_rowcount[2]}): mean≈{carts_rowcount[0]:.2f}, max≈{carts_rowcount[1]:.2f}")

# catalogue observations
if catalogue_rowcount or catalogue_error:
    parts.append("Catalogue logs:")
    if catalogue_rowcount:
        parts.append(f" - log.row_count (count={catalogue_rowcount[2]}): mean≈{catalogue_rowcount[0]:.2f}, max≈{catalogue_rowcount[1]:.2f}")
    if catalogue_error:
        parts.append(f" - log.error_count: mean≈{catalogue_error[0]:.2f}, max≈{catalogue_error[1]:.2f} (zeros observed)")

# rabbitmq note
if rabbit_cpu or rabbit_socket:
    parts.append("RabbitMQ metrics:")
    if rabbit_cpu:
        parts.append(f" - CPU mean≈{rabbit_cpu[0]:.2f}, max≈{rabbit_cpu[1]:.2f}")
    if rabbit_socket:
        parts.append(f" - Socket mean≈{rabbit_socket[0]:.2f}, max≈{rabbit_socket[1]:.2f}")

# General log.errors
parts.append("Log error counts are largely zero across observed components (no obvious error spikes).")
parts.append("Top observations to investigate: elevated 'carts' socket and memory values, and high 'catalogue' log.row_count (high request/row throughput).")

summary = "\n".join(parts)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  count    min(value)    max(value)   mean(value)    p95(value)
0          carts         cpu     25  1.814282e+00  6.543827e+00  3.721455e+00  6.184999e+00
1          carts  latency-50     25  9.661184e-03  1.771939e-02  1.294582e-02  1.561332e-02
2          carts  latency-90     25  2.186839e-02  3.724922e-02  2.383555e-02  2.566673e-02
3          carts         mem     25  1.910047e+08  2.069700e+08  2.023666e+08  2.067263e+08
4          carts      socket     25  9.450000e+00  1.393333e+01  1.124185e+01  1.377926e+01
..           ...         ...    ...           ...           ...           ...           ...
45  queue-master      socket     25  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00
46      rabbitmq         cpu     25  8.040535e-01  9.810124e-01  9.041514e-01  9.667465e-01
47      rabbitmq      diskio     25  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
48      rabbitmq         mem     25  8.948122e+07  9.284540e+07  9.161026e+07  9.284141e+07
49      rabbitmq      socket     25  1.100000e+01  1.100000e+01  1.100000e+01  1.100000e+01

[50 rows x 7 columns], Empty DataFrame
Columns: [cmdb_id, trace_name, count, min(value), max(value), mean(value), p95(value)]
Index: [],        cmdb_id         log_name  count  min(value)  max(value)  mean(value)  p95(value)
0        carts  log.error_count     25           0           0         0.00        0.00
1        carts    log.row_count     25          25         150        57.08      100.00
5    catalogue    log.row_count     25          17         169       154.68      168.00
4    catalogue  log.error_count     25           0           0         0.00        0.00
6    front-end  log.error_count     25           0           0         0.00        0.00
..         ...              ...    ...         ...         ...          ...         ...
17  session-db    log.row_count      5           5           5         5.00        5.00
22     user-db  log.error_count      4           0           0         0.00        0.00
23     user-db    log.row_count      4           1           6         2.25        5.25
10   orders-db  log.error_count      1           0           0         0.00        0.00
11   orders-db    log.row_count      1           1           1         1.00        1.00

[24 rows x 7 columns], 0                 carts
1              carts-db
2             catalogue
3          catalogue-db
4             front-end
5                orders
6             orders-db
7               payment
8          queue-master
9              rabbitmq
10    rabbitmq-exporter
11           session-db
12             shipping
13                 user
14              user-db
Name: cmdb_id, dtype: object)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing dataframes if present, otherwise read from CSVs
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")
    log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

# Ensure timestamps parsed and 'value' numeric
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# Components to restrict
components = ['carts','catalogue','orders','payment','user']

# Incident window in UTC
window_start = pd.to_datetime("2024-01-20 20:08:53", utc=True)
window_end   = pd.to_datetime("2024-01-20 20:38:53", utc=True)

def analyze_series(df, id_col, name_col, value_col, metric_type_label):
    rows = []
    # restrict to components
    sub_df = df[df[id_col].isin(components)]
    groups = sub_df.groupby([id_col, name_col])
    for (cmdb, name), grp in groups:
        vals_all = grp[value_col].dropna()
        if vals_all.empty:
            # skip series with no numeric data
            continue
        # Global thresholds computed on full series BEFORE window filtering
        global_p95 = float(vals_all.quantile(0.95))
        global_p90 = float(vals_all.quantile(0.90))
        global_p15 = float(vals_all.quantile(0.15))
        global_p5  = float(vals_all.quantile(0.05))
        # Filter to incident window
        mask = (grp['timestamp'] >= window_start) & (grp['timestamp'] <= window_end)
        window_df = grp.loc[mask].sort_values('timestamp')
        window_count = int(window_df.shape[0])
        window_max_value = float(window_df[value_col].max()) if window_count>0 else np.nan
        window_mean_value = float(window_df[value_col].mean()) if window_count>0 else np.nan
        # Breaches: above p95 and below p15 within window
        above_df = window_df[window_df[value_col] > global_p95].sort_values('timestamp')
        below_df = window_df[window_df[value_col] < global_p15].sort_values('timestamp')
        num_above = int(above_df.shape[0])
        earliest_above = (above_df['timestamp'].min().isoformat() if num_above>0 else pd.NA)
        num_below = int(below_df.shape[0])
        earliest_below = (below_df['timestamp'].min().isoformat() if num_below>0 else pd.NA)
        # Sample up to top 10 breach rows (either above p95 or below p15), sorted by timestamp
        breach_df = window_df[(window_df[value_col] > global_p95) | (window_df[value_col] < global_p15)]
        breach_sample = breach_df.sort_values('timestamp').head(10)
        # Represent sample as list of "ISOtimestamp|value" strings for compactness
        sample_list = []
        for _, r in breach_sample.iterrows():
            ts_iso = r['timestamp'].isoformat()
            sample_list.append(f"{ts_iso}|{r[value_col]}")
        rows.append({
            'cmdb_id': cmdb,
            'kpi_or_log_name': name,
            'metric_type': metric_type_label,
            'global_p95': global_p95,
            'global_p90': global_p90,
            'global_p15': global_p15,
            'global_p5': global_p5,
            'window_count': window_count,
            'window_max_value': window_max_value,
            'window_mean_value': window_mean_value,
            'num_points_above_p95': num_above,
            'earliest_timestamp_above_p95': earliest_above,
            'num_points_below_p15': num_below,
            'earliest_timestamp_below_p15': earliest_below,
            'breach_samples': sample_list
        })
    return pd.DataFrame(rows)

# Analyze metrics (cmdb_id, kpi_name)
metric_results = analyze_series(metric_df, 'cmdb_id', 'kpi_name', 'value', 'metric')

# Analyze logs (cmdb_id, log_name)
log_results = analyze_series(log_df, 'cmdb_id', 'log_name', 'value', 'log')

# Combine results
combined = pd.concat([metric_results, log_results], ignore_index=True)

# Keep output compact: sort by window_count desc then num_points_above_p95 desc, limit rows to top 50
combined = combined.sort_values(['window_count','num_points_above_p95'], ascending=[False, False]).reset_index(drop=True)
combined_head = combined.head(50)

# Display compact table with selected columns
cols = ['cmdb_id','kpi_or_log_name','metric_type','global_p95','global_p90','global_p15','global_p5',
        'window_count','window_max_value','window_mean_value',
        'num_points_above_p95','earliest_timestamp_above_p95',
        'num_points_below_p15','earliest_timestamp_below_p15','breach_samples']
combined_head[cols]

```
Out[2]:
```
Summary (plain English):

- Primary suspect: the "carts" service. Multiple metric KPIs for carts show breaches inside the incident window (2024-01-20 20:08:53 → 2024-01-20 20:38:53 UTC).
  - carts.cpu: global P95 ≈ 6.185. In-window: 25 points, 2 points above P95 (earliest above at 2024-01-20T20:11:00Z) and 4 points below P15 (earliest below at 2024-01-20T20:27:00Z).
  - carts.latency-50: global P95 ≈ 0.0156. In-window: 25 points, 2 above P95 (earliest 2024-01-20T20:11:00Z) and 4 below P15 (earliest 2024-01-20T20:18:00Z).
  - carts.latency-90: global P95 ≈ 0.0257. In-window: 25 points, 2 above P95 (earliest 2024-01-20T20:11:00Z) and 4 below P15 (earliest 2024-01-20T20:17:00Z).
  - carts.mem: global P95 ≈ 206,726,300. In-window: 25 points, 2 above P95 (earliest 2024-01-20T20:34:00Z) and 4 below P15 (earliest 2024-01-20T20:11:00Z).
  - carts.socket: global P95 ≈ 13.779. In-window: 25 points, 2 above P95 (earliest 2024-01-20T20:25:00Z) and 4 below P15 (earliest 2024-01-20T20:12:00Z).

- Other components (catalogue, orders, payment, user) show no error-count spikes in the window (log.error_count is zero across these services) and generally no notable breach activity reported in this analysis.

- Conclusion / recommended next steps:
  1. Investigate the carts service first — look at instances around 2024-01-20T20:11:00Z and later (20:25–20:34Z) where metrics exceed their historical P95s and several points fall below P15.
  2. Focus on CPU, memory, socket behavior and latency changes for carts; these simultaneous anomalies suggest a service-level issue (resource saturation, networking/socket problems, or GC/heap events).
  3. Check service logs and traces for carts for the timestamps above; other services show no error-count signals to indicate they are the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  kpi_or_log_name metric_type    global_p95    global_p90    global_p15     global_p5  window_count  window_max_value  window_mean_value  num_points_above_p95 earliest_timestamp_above_p95  num_points_below_p15 earliest_timestamp_below_p15                                     breach_samples
0       carts              cpu      metric  6.184999e+00  6.094638e+00  2.255604e+00  1.862427e+00            25      6.543827e+00       3.721455e+00                     2    2024-01-20T20:11:00+00:00                     4    2024-01-20T20:27:00+00:00  [2024-01-20T20:11:00+00:00|6.20505576894015, 2...
1       carts       latency-50      metric  1.561332e-02  1.447127e-02  1.094218e-02  9.884493e-03            25      1.771939e-02       1.294582e-02                     2    2024-01-20T20:11:00+00:00                     4    2024-01-20T20:18:00+00:00  [2024-01-20T20:11:00+00:00|0.0177193933751651,...
2       carts       latency-90      metric  2.566673e-02  2.438583e-02  2.243029e-02  2.204089e-02            25      3.724922e-02       2.383555e-02                     2    2024-01-20T20:11:00+00:00                     4    2024-01-20T20:17:00+00:00  [2024-01-20T20:11:00+00:00|0.0372492151451054,...
3       carts              mem      metric  2.067263e+08  2.063154e+08  1.982720e+08  1.927551e+08            25      2.069700e+08       2.023666e+08                     2    2024-01-20T20:34:00+00:00                     4    2024-01-20T20:11:00+00:00  [2024-01-20T20:11:00+00:00|191004672.0, 2024-0...
4       carts           socket      metric  1.377926e+01  1.361111e+01  9.710000e+00  9.563333e+00            25      1.393333e+01       1.124185e+01                     2    2024-01-20T20:25:00+00:00                     4    2024-01-20T20:12:00+00:00  [2024-01-20T20:12:00+00:00|9.7, 2024-01-20T20:...
..        ...              ...         ...           ...           ...           ...           ...           ...               ...                ...                   ...                          ...                   ...                          ...                                                ...
36  catalogue  log.error_count         log  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00            25      0.000000e+00       0.000000e+00                     0                         <NA>                     0                         <NA>                                                 []
37     orders  log.error_count         log  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00            25      0.000000e+00       0.000000e+00                     0                         <NA>                     0                         <NA>                                                 []
38    payment  log.error_count         log  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00            25      0.000000e+00       0.000000e+00                     0                         <NA>                     0                         <NA>                                                 []
39       user  log.error_count         log  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00            25      0.000000e+00       0.000000e+00                     0                         <NA>                     0                         <NA>                                                 []
40       user           diskio      metric  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00            24      0.000000e+00       0.000000e+00                     0                         <NA>                     0                         <NA>                                                 []

[41 rows x 15 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing DataFrames if present; otherwise read
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")
    log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

try:
    error_df
except NameError:
    error_df = pd.read_csv("error_logs.csv")
    error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# Ensure timestamps and numeric values
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# Focus on carts component
component = 'carts'

# Incident window
window_start = pd.to_datetime("2024-01-20 20:08:53", utc=True)
window_end   = pd.to_datetime("2024-01-20 20:38:53", utc=True)

# 1) Compute global P95 and P15 for each carts KPI using full available series
carts_full = metric_df[metric_df['cmdb_id'] == component].copy()
# If no carts metrics, produce empty results
if carts_full.empty:
    carts_thresholds = pd.DataFrame(columns=['kpi_name','global_p95','global_p15'])
else:
    carts_thresholds = carts_full.groupby('kpi_name')['value'].agg(
        global_p95=lambda s: float(s.quantile(0.95)),
        global_p15=lambda s: float(s.quantile(0.15))
    ).reset_index()

# 2) Filter metric rows for carts to the incident window
carts_window = carts_full[(carts_full['timestamp'] >= window_start) & (carts_full['timestamp'] <= window_end)].copy()
carts_window = carts_window.sort_values(['kpi_name','timestamp']).reset_index(drop=True)

# 3) Identify consecutive anomaly runs per KPI in window where value > P95 or value < P15
runs = []
for _, row in carts_thresholds.iterrows():
    kpi = row['kpi_name']
    p95 = row['global_p95']
    p15 = row['global_p15']
    kdf = carts_window[carts_window['kpi_name'] == kpi].sort_values('timestamp').reset_index(drop=True)
    if kdf.empty:
        continue
    # Mark anomalies
    kdf['is_anom'] = (kdf['value'] > p95) | (kdf['value'] < p15)
    # Compute boundaries: new group when time gap != 1 minute OR is_anom changes
    time_gap = (kdf['timestamp'].diff() != pd.Timedelta(minutes=1)).fillna(True)
    is_anom_change = (kdf['is_anom'] != kdf['is_anom'].shift(1)).fillna(True)
    group_start = time_gap | is_anom_change
    kdf['group_id'] = group_start.cumsum()
    # Group and select groups where is_anom True and length >=2
    grouped = kdf.groupby('group_id')
    for gid, g in grouped:
        if not g['is_anom'].all():
            continue
        run_len = int(g.shape[0])
        if run_len < 2:
            continue
        run_start = g['timestamp'].min()
        run_end = g['timestamp'].max()
        run_max = float(g['value'].max())
        run_min = float(g['value'].min())
        run_mean = float(g['value'].mean())
        # Determine anomaly type: if any value > p95 -> high, else low
        if (g['value'] > p95).any():
            anom_type = 'high'
            denom = p95
            severity_pct = (run_max - p95)/p95 if p95 != 0 else np.nan
        else:
            anom_type = 'low'
            denom = p15
            severity_pct = (p15 - run_min)/p15 if p15 != 0 else np.nan
        runs.append({
            'kpi_name': kpi,
            'anomaly_type': anom_type,
            'run_start': run_start.isoformat(),
            'run_end': run_end.isoformat(),
            'run_length': run_len,
            'run_max_value': run_max,
            'run_min_value': run_min,
            'run_mean_value': run_mean,
            'global_p95': p95,
            'global_p15': p15,
            'severity_pct': float(severity_pct)
        })

runs_df = pd.DataFrame(runs)

# Limit to top 20 ordered by run_length desc then severity_pct desc
if not runs_df.empty:
    runs_df = runs_df.sort_values(['run_length','severity_pct'], ascending=[False, False]).reset_index(drop=True)
    runs_df_top = runs_df.head(20)
else:
    runs_df_top = runs_df.copy()

# 4a) Aggregate carts logs in the same window by log_name: count, min, max, mean
carts_logs_window = log_df[(log_df['cmdb_id'] == component) & (log_df['timestamp'] >= window_start) & (log_df['timestamp'] <= window_end)].copy()
if carts_logs_window.empty:
    carts_log_agg = pd.DataFrame(columns=['log_name','count','min(value)','max(value)','mean(value)'])
else:
    carts_log_agg = carts_logs_window.groupby('log_name')['value'].agg(
        count='count',
        min_value='min',
        max_value='max',
        mean_value='mean'
    ).reset_index().rename(columns={'min_value':'min(value)','max_value':'max(value)','mean_value':'mean(value)'})

# 4b) Return up to 20 error_logs rows for carts within the window with timestamp ISO UTC and message
carts_errors_window = error_df[(error_df['cmdb_id'] == component) & (error_df['timestamp'] >= window_start) & (error_df['timestamp'] <= window_end)].copy()
if not carts_errors_window.empty:
    carts_errors_window = carts_errors_window.sort_values('timestamp').reset_index(drop=True)
    # keep only timestamp and message and convert timestamp to ISO
    carts_errors_window = carts_errors_window.loc[:, ['timestamp','message']].copy()
    carts_errors_window['timestamp'] = carts_errors_window['timestamp'].dt.tz_convert('UTC').dt.isoformat()
    carts_errors_sample = carts_errors_window.head(20)
else:
    carts_errors_sample = carts_errors_window.copy()

# Final compact outputs: runs_df_top, carts_log_agg, carts_errors_sample
runs_df_top, carts_log_agg, carts_errors_sample

```
Out[3]:
```
Summary for carts (plain English):

Key anomaly runs detected in the incident window (2024-01-20 20:08:53 → 2024-01-20 20:38:53 UTC):

- A cluster of short high-value runs at ~2024-01-20T20:11:00Z (length = 2 minutes):
  - latency-90 (high): 20:11 → 20:12, length 2, run_max ≈ 0.03725s, run_mean ≈ 0.0316s — ~45% above its global P95 (most severe percentage).
  - workload (high): 20:11 → 20:12, length 2, run_max ≈ 10.267, run_mean ≈ 9.026 — ~20% above global P95.
  - latency-50 (high): 20:11 → 20:12, length 2, run_max ≈ 0.01772s, run_mean ≈ 0.01681s — ~13.5% above global P95.

- Multiple low-value runs (length >=2) later in the window:
  - latency-50 (low): 20:18 → 20:21, length 4, values below global P15 (severity ≈ 11.7% relative to P15).
  - mem (low): 20:11 → 20:14, length 4, memory values below global P15 (severity ≈ 3.7%).
  - latency-90 (low): 20:19 → 20:21, length 3 (severity ≈ 2.5%).

- Another mem high run near the end of the window:
  - mem (high): 20:34 → 20:35, length 2, small severity (~0.12% above P95).

Logs (carts) in the same window:
- log.error_count: 25 samples, min=0, max=0, mean=0.00 (no error-count spikes).
- log.row_count: 25 samples, min=25, max=150, mean≈57.08 (some variation in row throughput).

Error logs:
- No error_log entries for cmdb_id='carts' in the window (none returned).

Interpretation and recommended next steps:
- The strongest signal is a short burst of high values around 20:11–20:12 (especially latency-90 and workload). These simultaneous high anomalies suggest a transient load/spike or service-level event impacting latency and throughput.
- There are also sustained low-value runs (latency and memory) later in the window, indicating additional abnormal behavior (not just isolated spikes).
- Despite these metric anomalies, log.error_count is zero and no error log messages were found, so look beyond simple error logs: examine application traces, request patterns, GC/heap activity, and external load at 20:11–20:12; check resource metrics (CPU/socket/disk I/O) and upstream traffic that could explain the workload/latency spike.
- Prioritize investigating carts around 20:11–20:12, then the subsequent low-value periods and the mem fluctuation at ~20:34.

The original code execution output of IPython Kernel is also provided below for reference:

(     kpi_name anomaly_type                  run_start                    run_end  run_length  run_max_value  run_min_value  run_mean_value    global_p95    global_p15  severity_pct
0  latency-50          low  2024-01-20T20:18:00+00:00  2024-01-20T20:21:00+00:00           4   1.076985e-02   9.661184e-03    1.017631e-02  1.561332e-02  1.094218e-02      0.117070
1         mem          low  2024-01-20T20:11:00+00:00  2024-01-20T20:14:00+00:00           4   1.973035e+08   1.910047e+08    1.938747e+08  2.067263e+08  1.982720e+08      0.036653
2  latency-90          low  2024-01-20T20:19:00+00:00  2024-01-20T20:21:00+00:00           3   2.233226e-02   2.186839e-02    2.205623e-02  2.566673e-02  2.243029e-02      0.025051
3  latency-90         high  2024-01-20T20:11:00+00:00  2024-01-20T20:12:00+00:00           2   3.724922e-02   2.594901e-02    3.159911e-02  2.566673e-02  2.243029e-02      0.451264
4    workload         high  2024-01-20T20:11:00+00:00  2024-01-20T20:12:00+00:00           2   1.026700e+01   7.784217e+00    9.025608e+00  8.543227e+00  7.912470e+00      0.201771
5  latency-50         high  2024-01-20T20:11:00+00:00  2024-01-20T20:12:00+00:00           2   1.771939e-02   1.589540e-02    1.680739e-02  1.561332e-02  1.094218e-02      0.134890
6         mem         high  2024-01-20T20:34:00+00:00  2024-01-20T20:35:00+00:00           2   2.069700e+08   2.068180e+08    2.068940e+08  2.067263e+08  1.982720e+08      0.001179,           log_name  count  min(value)  max(value)  mean(value)
0  log.error_count     25           0           0         0.00
1    log.row_count     25          25         150        57.08, Empty DataFrame
Columns: [timestamp, cmdb_id, message, error]
Index: [])```
```